# 🔄 Python `deque` — The Master Guide
### *From Zero to Interview-Ready*

---

> **Mental Model First:**  
> A `deque` is a **double-ended queue** — think of it as a **hallway** with two open doors.  
> You can push or pop from the **LEFT door** or the **RIGHT door** — both ends are live.  
> A regular Python `list` only has one "live" end efficiently (the right).  
> A `deque` makes **both ends O(1)**.

---

## 📋 Table of Contents

| # | Section |
|---|---|
| 1 | [What Is a Deque? The Visual Model](#1) |
| 2 | [Creating a Deque](#2) |
| 3 | [The Core API — All Operations](#3) |
| 4 | [LEFT vs RIGHT — Your Decision Map](#4) |
| 5 | [Rotation — The Unique Superpower](#5) |
| 6 | [Maxlen — The Sliding Window Mode](#6) |
| 7 | [Deque vs List — Performance Truth Table](#7) |
| 8 | [Pattern 1: BFS Queue](#8) |
| 9 | [Pattern 2: Sliding Window Maximum (Monotonic Deque)](#9) |
| 10 | [Pattern 3: Fixed-Size History / Recent-K Buffer](#10) |
| 11 | [Pattern 4: Palindrome Checker](#11) |
| 12 | [Pattern 5: Task Scheduler / Round Robin](#12) |
| 13 | [Pattern 6: Undo Stack (Double-Ended History)](#13) |
| 14 | [Interview Cheat Sheet](#14) |

<a id='1'></a>
## 1. 🏗️ What Is a Deque? The Visual Model

---

```
                    DEQUE HALLWAY

  LEFT DOOR                              RIGHT DOOR
  (index 0)                             (index -1)
     │                                       │
     ▼                                       ▼
  ┌──────┬──────┬──────┬──────┬──────┬──────┐
  │  'A' │  'B' │  'C' │  'D' │   'E' │  'F' │
  └──────┴──────┴──────┴──────┴──────┴──────┘
     ▲                                       ▲
     │                                       │
  appendleft()                           append()
  popleft()                              pop()

  Both ends are O(1).  No penalty for left-side ops.
  This is the entire reason deque exists.
```

---

### Why does this matter?

In a Python `list`, doing `list.insert(0, x)` or `list.pop(0)` is **O(n)** — every element shifts.  
In a `deque`, `appendleft()` and `popleft()` are **O(1)** — nothing shifts.  

The deque is implemented as a **doubly-linked list of fixed-size blocks** internally.  
Left and right pointers move freely without touching other elements.

<a id='2'></a>
## 2. 🔧 Creating a Deque

In [1]:
from collections import deque

# --- Empty deque ---
d1 = deque()
print("Empty:", d1)

# --- From a list ---
d2 = deque([10, 20, 30, 40, 50])
print("From list:", d2)

# --- From a string (each char becomes an element) ---
d3 = deque("hello")
print("From string:", d3)

# --- From a range ---
d4 = deque(range(5))
print("From range:", d4)

# --- With a max size (sliding window / recent-K buffer) ---
d5 = deque([1, 2, 3, 4, 5], maxlen=3)
print("With maxlen=3 (overflow auto-evicts):", d5)

Empty: deque([])
From list: deque([10, 20, 30, 40, 50])
From string: deque(['h', 'e', 'l', 'l', 'o'])
From range: deque([0, 1, 2, 3, 4])
With maxlen=3 (overflow auto-evicts): deque([3, 4, 5], maxlen=3)


<a id='3'></a>
## 3. 🛠️ The Core API — Every Operation

---

```
OPERATION          SIDE    COMPLEXITY   WHAT IT DOES
─────────────────────────────────────────────────────────
append(x)          RIGHT   O(1)         Add x to the right end
appendleft(x)      LEFT    O(1)         Add x to the left end
pop()              RIGHT   O(1)         Remove & return rightmost element
popleft()          LEFT    O(1)         Remove & return leftmost element
extend(iter)       RIGHT   O(k)         Add all items from iterable to right
extendleft(iter)   LEFT    O(k)         Add all items to left (REVERSED order!)
rotate(n)          BOTH    O(n)         Rotate right by n (neg = rotate left)
reverse()          BOTH    O(n)         Reverse in place
count(x)           BOTH    O(n)         Count occurrences of x
index(x)           BOTH    O(n)         Find index of x
remove(x)          BOTH    O(n)         Remove first occurrence of x
insert(i, x)       BOTH    O(n)         Insert x at position i
clear()            BOTH    O(n)         Remove all elements
copy()             BOTH    O(n)         Shallow copy
len(d)             —       O(1)         Number of elements
d[0]               LEFT    O(1)         Peek left end
d[-1]              RIGHT   O(1)         Peek right end
d[i]               MIDDLE  O(n)         Random access — NOT fast like a list!
─────────────────────────────────────────────────────────
⚠️  Random access d[i] for middle elements is O(n).
    Only the two ends are O(1). Keep that in mind.
```

In [2]:
from collections import deque

d = deque([10, 20, 30])
print("Start:", d)

# -- Appending --
d.append(40)          # add right
print("After append(40):", d)

d.appendleft(0)       # add left
print("After appendleft(0):", d)

# -- Popping --
right_val = d.pop()
print(f"pop() returned {right_val}, deque now:", d)

left_val = d.popleft()
print(f"popleft() returned {left_val}, deque now:", d)

# -- Peeking (no removal) --
print("Peek LEFT  d[0]:", d[0])
print("Peek RIGHT d[-1]:", d[-1])

# -- Extending --
d.extend([100, 200])          # adds to right in order
print("After extend([100,200]):", d)

d.extendleft([7, 8])          # adds to LEFT — note reversed!
print("After extendleft([7,8]):", d)  # 8 then 7 at left

Start: deque([10, 20, 30])
After append(40): deque([10, 20, 30, 40])
After appendleft(0): deque([0, 10, 20, 30, 40])
pop() returned 40, deque now: deque([0, 10, 20, 30])
popleft() returned 0, deque now: deque([10, 20, 30])
Peek LEFT  d[0]: 10
Peek RIGHT d[-1]: 30
After extend([100,200]): deque([10, 20, 30, 100, 200])
After extendleft([7,8]): deque([8, 7, 10, 20, 30, 100, 200])


### ⚠️ `extendleft` Gotcha — Why Is It Reversed?

```
extendleft([7, 8])  processes: 7 first, then 8

Step 1: appendleft(7)  →  [7, ...rest...]
Step 2: appendleft(8)  →  [8, 7, ...rest...]

So the result is [8, 7] at the left — reversed from your input list.
This is NOT a bug. It's by design. Just keep it in mind.
```

<a id='4'></a>
## 4. 🗺️ LEFT vs RIGHT — Your Decision Map

---

```
  WHAT ARE YOU BUILDING?         USE THIS END
  ─────────────────────────────────────────────────────────
  Standard queue (FIFO)          append() RIGHT in,
  "first in, first out"          popleft() LEFT out

  Stack (LIFO)                   append() RIGHT in,
  "last in, first out"           pop() RIGHT out
                                 (or appendleft + popleft)

  Sliding window — evict old     appendleft() or append()
  entries as window slides       depends on direction of slide

  BFS tree traversal             append() to enqueue children
                                 popleft() to process nodes

  Palindrome check               popleft() and pop() together
                                 compare LEFT vs RIGHT char

  Round-robin / task rotation    rotate(1) or rotate(-1)

  Recent-K history buffer        maxlen=K + append() right
                                 auto-evicts from LEFT
  ─────────────────────────────────────────────────────────
```

---

### The FIFO Queue Pattern (Most Common)

```
  ENQUEUE →  append()  RIGHT end
  DEQUEUE ←  popleft() LEFT end

  [  A  |  B  |  C  |  D  ]
   ←pop                 ←push new items
  popleft()             append()

  This is BFS. This is task queues. This is the bread and butter.
```

In [3]:
from collections import deque

# FIFO Queue demo
queue = deque()

# Enqueue jobs
for job in ["Job-A", "Job-B", "Job-C", "Job-D"]:
    queue.append(job)        # push RIGHT
    print(f"Enqueued {job}: {queue}")

print()

# Process jobs FIFO
while queue:
    current = queue.popleft()  # pop LEFT — oldest first
    print(f"Processing: {current}  |  remaining: {queue}")

Enqueued Job-A: deque(['Job-A'])
Enqueued Job-B: deque(['Job-A', 'Job-B'])
Enqueued Job-C: deque(['Job-A', 'Job-B', 'Job-C'])
Enqueued Job-D: deque(['Job-A', 'Job-B', 'Job-C', 'Job-D'])

Processing: Job-A  |  remaining: deque(['Job-B', 'Job-C', 'Job-D'])
Processing: Job-B  |  remaining: deque(['Job-C', 'Job-D'])
Processing: Job-C  |  remaining: deque(['Job-D'])
Processing: Job-D  |  remaining: deque([])


<a id='5'></a>
## 5. 🔄 Rotation — The Unique Superpower

---

```
  rotate(n)   →  rotate RIGHT by n steps  (elements wrap around from right to left)
  rotate(-n)  →  rotate LEFT  by n steps  (elements wrap around from left to right)

  BEFORE:  [ A | B | C | D | E ]

  rotate(2):
  Step 1: pop D from right, appendleft D  →  [ D | A | B | C | E ]
  Step 2: pop E from right, appendleft E  →  [ E | D | A | B | C ]
  AFTER:   [ E | D | A | B | C ]

  rotate(-2):
  Step 1: popleft A, append A  →  [ B | C | D | E | A ]
  Step 2: popleft B, append B  →  [ C | D | E | A | B ]
  AFTER:   [ C | D | E | A | B ]

  Think of it as spinning a ring — elements wrap around.
  No element is created or destroyed, just repositioned.
```

In [4]:
from collections import deque

d = deque(['A', 'B', 'C', 'D', 'E'])
print("Original:    ", list(d))

d.rotate(2)
print("rotate(2):   ", list(d))   # RIGHT rotation — last 2 wrap to front

d.rotate(-2)
print("rotate(-2):  ", list(d))   # Back to original

d.rotate(1)
print("rotate(1):   ", list(d))   # Last element wraps to front

d.rotate(-1)
print("rotate(-1):  ", list(d))   # Front element wraps to back

Original:     ['A', 'B', 'C', 'D', 'E']
rotate(2):    ['D', 'E', 'A', 'B', 'C']
rotate(-2):   ['A', 'B', 'C', 'D', 'E']
rotate(1):    ['E', 'A', 'B', 'C', 'D']
rotate(-1):   ['A', 'B', 'C', 'D', 'E']


<a id='6'></a>
## 6. 📐 `maxlen` — The Sliding Window Mode

---

```
  deque(maxlen=3) with incoming stream: [1, 2, 3, 4, 5]

  append(1)  →  [ 1 ]          (room for 2 more)
  append(2)  →  [ 1 | 2 ]      (room for 1 more)
  append(3)  →  [ 1 | 2 | 3 ]  (full)
  append(4)  →  [ 2 | 3 | 4 ]  ← 1 auto-evicted from LEFT
  append(5)  →  [ 3 | 4 | 5 ]  ← 2 auto-evicted from LEFT

  The deque AUTOMATICALLY evicts the oldest (leftmost) element
  when a new one is appended to the right and it's full.

  This is a fixed-size window with zero manual eviction code.
  You never write: if len(d) > k: d.popleft()
  The deque handles it for you.
```

In [5]:
from collections import deque

# Simulate a data stream — keep only last 3 readings
window = deque(maxlen=3)
stream = [10, 25, 8, 42, 17, 31, 5]

print(f"{'Incoming':>10}  {'Window':30}  {'Current Max':<12}")
print("-" * 56)

for val in stream:
    window.append(val)           # auto-evicts if over maxlen
    print(f"{val:>10}  {str(list(window)):30}  {max(window):<12}")

print()
print("Final window:", list(window))
print("maxlen is fixed:", window.maxlen)

  Incoming  Window                          Current Max 
--------------------------------------------------------
        10  [10]                            10          
        25  [10, 25]                        25          
         8  [10, 25, 8]                     25          
        42  [25, 8, 42]                     42          
        17  [8, 42, 17]                     42          
        31  [42, 17, 31]                    42          
         5  [17, 31, 5]                     31          

Final window: [17, 31, 5]
maxlen is fixed: 3


<a id='7'></a>
## 7. ⚡ Deque vs List — Performance Truth Table

---

```
  OPERATION              LIST        DEQUE       WINNER
  ────────────────────────────────────────────────────────
  append(x)              O(1)*       O(1)        Tie
  pop()                  O(1)        O(1)        Tie
  appendleft / insert(0) O(n) ❌     O(1) ✅     DEQUE
  popleft / pop(0)       O(n) ❌     O(1) ✅     DEQUE
  Random access d[i]     O(1) ✅     O(n) ❌     LIST
  len()                  O(1)        O(1)        Tie
  in (membership test)   O(n)        O(n)        Tie
  Sorting                O(n log n)  ❌ not fast  LIST
  Slicing d[1:4]         O(k) ✅     ❌ manual   LIST
  Memory                 ~compact    ~8x block   LIST (slightly)
  ────────────────────────────────────────────────────────
  * amortized

  RULE OF THUMB:
  → If you need fast left-side ops → USE DEQUE
  → If you need random access / slicing → USE LIST
  → BFS, sliding window, queues → DEQUE every time
```

In [6]:
import time
from collections import deque

N = 100_000

# --- List insert at front (O(n) per op) ---
lst = []
start = time.perf_counter()
for i in range(N):
    lst.insert(0, i)     # O(n) — shifts everything
list_time = time.perf_counter() - start

# --- Deque appendleft (O(1) per op) ---
dq = deque()
start = time.perf_counter()
for i in range(N):
    dq.appendleft(i)     # O(1) — no shifting
deque_time = time.perf_counter() - start

print(f"Inserting {N:,} items at the LEFT end:")
print(f"  list.insert(0, x):  {list_time:.4f}s")
print(f"  deque.appendleft(): {deque_time:.4f}s")
print(f"  Deque is {list_time/deque_time:.1f}x faster")

Inserting 100,000 items at the LEFT end:
  list.insert(0, x):  2.9924s
  deque.appendleft(): 0.0088s
  Deque is 338.4x faster


<a id='8'></a>
## 8. 🌐 Pattern 1: BFS Queue

---

BFS (Breadth-First Search) is the canonical deque use case.  
You explore nodes **level by level** — FIFO order.

```
  GRAPH:
        A
       / \
      B   C
     / \   \
    D   E   F

  BFS visit order: A → B → C → D → E → F

  QUEUE state at each step:
  Start:        [ A ]
  Pop A, add B,C: [ B | C ]
  Pop B, add D,E: [ C | D | E ]
  Pop C, add F:   [ D | E | F ]
  Pop D:          [ E | F ]
  Pop E:          [ F ]
  Pop F:          []
```

In [7]:
from collections import deque

# Graph as adjacency list
graph = {
    'A': ['B', 'C'],
    'B': ['D', 'E'],
    'C': ['F'],
    'D': [],
    'E': [],
    'F': []
}

def bfs(graph, start):
    visited = set()
    queue = deque([start])    # seed the queue
    visited.add(start)
    order = []

    while queue:
        node = queue.popleft()   # FIFO: oldest first (LEFT)
        order.append(node)
        print(f"  Visit: {node}  |  queue: {list(queue)}")

        for neighbor in graph[node]:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)   # push RIGHT

    return order

print("BFS traversal from 'A':")
result = bfs(graph, 'A')
print()
print("Visit order:", result)

BFS traversal from 'A':
  Visit: A  |  queue: []
  Visit: B  |  queue: ['C']
  Visit: C  |  queue: ['D', 'E']
  Visit: D  |  queue: ['E', 'F']
  Visit: E  |  queue: ['F']
  Visit: F  |  queue: []

Visit order: ['A', 'B', 'C', 'D', 'E', 'F']


<a id='9'></a>
## 9. 🪟 Pattern 2: Sliding Window Maximum (Monotonic Deque)

---

This is **LC #239** — the hardest deque pattern and the most interview-relevant.

```
  GOAL: For each window position, find the maximum in O(1).

  KEY IDEA: The deque stores INDICES, not values.
            It is kept in DECREASING VALUE ORDER — a monotonic deque.
            The LEFTMOST index always holds the current window max.

  INVARIANTS (both must hold at all times):
  1. Deque only holds indices within the current window.
     → Evict from LEFT if index is out of window.
  2. Deque is monotonically DECREASING by value.
     → Before appending new index, evict from RIGHT
        any index whose value is ≤ new value.
        (Those can never be the max while new val is in window.)

  nums = [1, 3, -1, -3, 5, 3, 6, 7], k=3

  i=0  val=1   dq=[0]           window [1]       (no result yet)
  i=1  val=3   evict 0 (1<3)    dq=[1]           (no result yet)
  i=2  val=-1  keep              dq=[1,2]  max=nums[1]=3
  i=3  val=-3  keep              dq=[1,2,3] max=nums[1]=3
  i=4  val=5   evict 2,3 (≤5)   dq=[4]    max=nums[4]=5   ← window slid past i=1
  i=5  val=3   keep              dq=[4,5]  max=nums[4]=5
  i=6  val=6   evict 4,5 (≤6)   dq=[6]    max=nums[6]=6
  i=7  val=7   evict 6 (≤7)     dq=[7]    max=nums[7]=7

  Result: [3, 3, 5, 5, 6, 7]
```

In [8]:
from collections import deque

def sliding_window_max(nums, k):
    """
    LC #239. Sliding window maximum.
    Deque stores INDICES, maintained in decreasing-value order.
    """
    dq = deque()   # stores indices
    result = []

    for i, val in enumerate(nums):

        # 1. Evict LEFT: out-of-window indices
        if dq and dq[0] < i - k + 1:
            dq.popleft()

        # 2. Evict RIGHT: useless smaller values
        while dq and nums[dq[-1]] <= val:
            dq.pop()

        dq.append(i)

        # 3. Record max once window is full
        if i >= k - 1:
            result.append(nums[dq[0]])   # leftmost = current max

    return result


def run_tests(fn):
    tests = [
        ([1, 3, -1, -3, 5, 3, 6, 7], 3, [3, 3, 5, 5, 6, 7]),
        ([1],                          1, [1]),
        ([9, 8, 7, 6],                 2, [9, 8, 7]),
        ([1, 2, 3, 4],                 2, [2, 3, 4]),
    ]
    for nums, k, expected in tests:
        got = fn(nums, k)
        status = "✅" if got == expected else "❌"
        print(f"{status}  nums={nums}, k={k}")
        if got != expected:
            print(f"   Expected: {expected}")
            print(f"   Got:      {got}")

run_tests(sliding_window_max)

✅  nums=[1, 3, -1, -3, 5, 3, 6, 7], k=3
✅  nums=[1], k=1
✅  nums=[9, 8, 7, 6], k=2
✅  nums=[1, 2, 3, 4], k=2


<a id='10'></a>
## 10. 📼 Pattern 3: Fixed-Size History / Recent-K Buffer

---

Use `maxlen=K`. The deque auto-evicts. No manual pruning.

```
  Use cases:
  - Last K web pages visited (browser history)
  - Last K telemetry readings (your Citi work!)
  - Last K commands in a terminal
  - Rolling average / rolling stats
```

In [9]:
from collections import deque

def rolling_average(stream, k):
    """
    Compute rolling average over a stream using maxlen deque.
    No index math. No manual eviction. Just append and average.
    """
    window = deque(maxlen=k)
    averages = []

    for val in stream:
        window.append(val)                    # auto-evicts oldest
        avg = sum(window) / len(window)
        averages.append(round(avg, 2))

    return averages


# Simulate server latency readings (ms)
latency_stream = [120, 145, 98, 200, 175, 130, 88, 210, 95, 160]
k = 3

avgs = rolling_average(latency_stream, k)

print(f"Rolling average (window={k}):")
print(f"{'Reading':>10}  {'Rolling Avg':>12}")
print("-" * 25)
for reading, avg in zip(latency_stream, avgs):
    print(f"{reading:>10}ms  {avg:>10}ms")

Rolling average (window=3):
   Reading   Rolling Avg
-------------------------
       120ms       120.0ms
       145ms       132.5ms
        98ms       121.0ms
       200ms      147.67ms
       175ms      157.67ms
       130ms      168.33ms
        88ms       131.0ms
       210ms      142.67ms
        95ms       131.0ms
       160ms       155.0ms


<a id='11'></a>
## 11. 🔤 Pattern 4: Palindrome Checker

---

Classic use of **simultaneously popping from both ends**.

```
  Word: "racecar"

  Deque:  [ r | a | c | e | c | a | r ]

  Round 1: popleft() = 'r'   pop() = 'r'   match ✅
  Round 2: popleft() = 'a'   pop() = 'a'   match ✅
  Round 3: popleft() = 'c'   pop() = 'c'   match ✅
  Round 4: only 'e' left (odd length) — done ✅
  → PALINDROME

  Word: "hello"
  Round 1: popleft() = 'h'   pop() = 'o'   NO MATCH ❌
  → NOT PALINDROME
```

In [10]:
from collections import deque

def is_palindrome(s):
    """
    Check if a string is a palindrome.
    Uses both ends simultaneously.
    """
    d = deque(s.lower())       # normalize case

    while len(d) > 1:          # need at least 2 chars to compare
        left  = d.popleft()   # grab LEFT
        right = d.pop()       # grab RIGHT
        if left != right:
            return False

    return True


def run_tests(fn):
    tests = [
        ("racecar", True),
        ("hello",   False),
        ("a",       True),
        ("aba",     True),
        ("abba",    True),
        ("abcd",    False),
        ("madam",   True),
    ]
    for word, expected in tests:
        got = fn(word)
        status = "✅" if got == expected else "❌"
        print(f"{status}  '{word}'  → {got}")

run_tests(is_palindrome)

✅  'racecar'  → True
✅  'hello'  → False
✅  'a'  → True
✅  'aba'  → True
✅  'abba'  → True
✅  'abcd'  → False
✅  'madam'  → True


<a id='12'></a>
## 12. ⚙️ Pattern 5: Task Scheduler / Round Robin

---

```
  ROTATE to move task to end after processing.
  No index math. No modulo. Just rotate(-1).

  Tasks: [A, B, C]

  Round 1:
    Process A (leftmost), rotate(-1)  →  [B, C, A]
  Round 2:
    Process B, rotate(-1)             →  [C, A, B]
  Round 3:
    Process C, rotate(-1)             →  [A, B, C]
  ... repeats

  Alternatively: popleft() → process → append() back
  Same result, more explicit.
```

In [11]:
from collections import deque

def round_robin_scheduler(tasks, cycles):
    """
    Simulate round-robin task scheduling.
    Each task gets one time slice per cycle.
    """
    queue = deque(tasks)
    log = []

    for cycle in range(1, cycles + 1):
        task = queue.popleft()        # grab current task
        log.append((cycle, task))
        queue.append(task)            # push to back of line

    return log


schedule = round_robin_scheduler(["Task-A", "Task-B", "Task-C"], cycles=7)

print("Round-Robin Schedule:")
print(f"{'Cycle':>6}  {'Task'}")
print("-" * 18)
for cycle, task in schedule:
    print(f"{cycle:>6}  {task}")

Round-Robin Schedule:
 Cycle  Task
------------------
     1  Task-A
     2  Task-B
     3  Task-C
     4  Task-A
     5  Task-B
     6  Task-C
     7  Task-A


<a id='13'></a>
## 13. ↩️ Pattern 6: Undo Stack (Double-Ended History)

---

```
  UNDO uses both ends for "undo" and "redo" in two separate deques.

  undo_stack:  stores past actions  →  pop() to undo
  redo_stack:  stores undone actions → pop() to redo

  Action:      type("hello")        undo_stack: [type("hello")]
  Action:      bold()               undo_stack: [type("hello"), bold()]
  Undo:        pop bold()           undo_stack: [type("hello")]  redo: [bold()]
  Undo:        pop type()           undo_stack: []               redo: [bold(), type()]
  Redo:        pop type() from redo undo_stack: [type()]         redo: [bold()]
```

In [12]:
from collections import deque

class TextEditor:
    """
    Simple text editor with undo/redo using deques.
    Both stacks grow from the right (append/pop).
    """

    def __init__(self):
        self.text = ""
        self.undo_stack = deque()   # history of (action, snapshot)
        self.redo_stack = deque()   # undone actions

    def type(self, chars):
        self.undo_stack.append(("type", self.text))  # save state before
        self.redo_stack.clear()     # new action wipes redo
        self.text += chars
        print(f"  type('{chars}') → '{self.text}'")

    def delete(self, n):
        self.undo_stack.append(("delete", self.text))
        self.redo_stack.clear()
        self.text = self.text[:-n]
        print(f"  delete({n})    → '{self.text}'")

    def undo(self):
        if not self.undo_stack:
            print("  undo: nothing to undo")
            return
        action, prev_text = self.undo_stack.pop()   # pop from RIGHT
        self.redo_stack.append((action, self.text)) # push to redo RIGHT
        self.text = prev_text
        print(f"  undo() → '{self.text}'")

    def redo(self):
        if not self.redo_stack:
            print("  redo: nothing to redo")
            return
        action, next_text = self.redo_stack.pop()
        self.undo_stack.append((action, self.text))
        self.text = next_text
        print(f"  redo() → '{self.text}'")


editor = TextEditor()
print("--- Actions ---")
editor.type("Hello")
editor.type(", World")
editor.delete(6)
print()
print("--- Undo ---")
editor.undo()
editor.undo()
print()
print("--- Redo ---")
editor.redo()
editor.undo()
editor.undo()

--- Actions ---
  type('Hello') → 'Hello'
  type(', World') → 'Hello, World'
  delete(6)    → 'Hello,'

--- Undo ---
  undo() → 'Hello, World'
  undo() → 'Hello'

--- Redo ---
  redo() → 'Hello, World'
  undo() → 'Hello'
  undo() → ''


<a id='14'></a>
## 14. 📋 Interview Cheat Sheet

---

### When to reach for `deque`:

| Signal in the Problem | What to Do |
|---|---|
| BFS / level-order traversal | `deque` as FIFO queue |
| Sliding window maximum/minimum | Monotonic deque (indices!) |
| "Last K" / recent history | `deque(maxlen=K)` |
| Process from both ends | `popleft()` + `pop()` |
| Circular / round-robin scheduling | `rotate(-1)` or popleft+append |
| Undo/redo history | Two deques as stacks |
| Palindrome check | popleft + pop comparison |

---

### The O(1) operations — memorize these:

```python
from collections import deque

d = deque()
d.append(x)       # RIGHT end — O(1)
d.appendleft(x)   # LEFT end  — O(1)
d.pop()           # RIGHT end — O(1)
d.popleft()       # LEFT end  — O(1)
d[0]              # PEEK left  — O(1)
d[-1]             # PEEK right — O(1)
len(d)            # size        — O(1)
```

---

### Gotchas to not forget:

```
❌  d[i] for middle elements is O(n) — not fast like a list
❌  extendleft([1,2,3]) adds them reversed — [3,2,1] at left
❌  deque is not sortable in place (convert to list to sort)
❌  No slicing: d[1:4] raises TypeError
✅  To convert: list(d)   or   sorted(d)
✅  maxlen auto-evicts — no manual if len > k checks needed
✅  deque is thread-safe for append/popleft (GIL-protected)
```

---

### Quick Pattern Templates:

```python
# FIFO Queue (BFS)
q = deque([start])
while q:
    node = q.popleft()
    for neighbor in graph[node]:
        q.append(neighbor)

# Monotonic deque (sliding window max)
dq = deque()   # stores indices
for i, val in enumerate(nums):
    if dq and dq[0] < i - k + 1: dq.popleft()   # out of window
    while dq and nums[dq[-1]] <= val: dq.pop()   # useless small vals
    dq.append(i)
    if i >= k - 1: result.append(nums[dq[0]])    # max is leftmost

# Recent-K buffer
window = deque(maxlen=K)
for val in stream:
    window.append(val)   # auto-evicts old

# Palindrome check
d = deque(s)
while len(d) > 1:
    if d.popleft() != d.pop(): return False
return True
```

---
## 🎯 Summary Map

```
                         PYTHON DEQUE
                        /            \
               LEFT END              RIGHT END
            appendleft()              append()
            popleft()                 pop()
               │                       │
        ┌──────┘                       └──────┐
        │                                     │
  BFS dequeue                         BFS enqueue
  Palindrome (left char)              Palindrome (right char)
  Monotonic: evict stale window       Monotonic: add new index
  Round-robin: take current           Round-robin: re-enqueue
        │                                     │
        └──────────────┬───────────────────────┘
                       │
                   BOTH ENDS
               rotate(n) / rotate(-n)
               maxlen auto-eviction
               palindrome compare
               undo/redo stacks
```

---
*End of Deque Master Guide — Sean Edition*